# IFA Modeling Test — Licata LNG (December 2023)

Google Colab / local notebook that rebuilds the Infrastructure Finance and Advisory modeling test in Python.

**Tasks**
1. Model project cash flows (ignore taxes, insurance, working capital).
2. Size senior debt at a **1.50x** minimum DSCR with a **6-year amortization grace** (first principal in 2030) and report:
   - Maximum debt the project can support
   - Sponsor levered IRR

Open in Colab from GitHub after this folder is pushed, or run the cells locally.

## 0. Setup

In Colab, clone this folder from your repo (update the URL if you publish it as a standalone repository).

In [ ]:
import sys
from pathlib import Path

# Local checkout (Cursor / laptop)
ROOT = Path.cwd()
if not (ROOT / "src" / "lng_model.py").exists():
    # Running from repo root
    candidate = ROOT / "ifa-modeling-colab"
    if (candidate / "src" / "lng_model.py").exists():
        ROOT = candidate

# Google Colab: clone if needed
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not (ROOT / "src" / "lng_model.py").exists():
    # Update REPO_URL after you create the standalone GitHub repo.
    REPO_URL = "https://github.com/daxsmordin-ai/Smordin-Capital.git"
    SUBDIR = "ifa-modeling-colab"
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Smordin-Capital"], check=True)
    ROOT = Path("/content/Smordin-Capital") / SUBDIR

sys.path.insert(0, str(ROOT))
print("Using project root:", ROOT)
print("In Colab:", IN_COLAB)

In [ ]:
%pip install -q numpy pandas matplotlib

import pandas as pd
import matplotlib.pyplot as plt

from src.lng_model import answers_markdown, run_model

result = run_model(draw_mode="prorata", capitalize_idc=True)
print(result.summary())

## 1. Project assumptions (from the test memo)

| Item | Value |
| --- | --- |
| Facility | 1.5 mtpa USGC LNG liquefaction |
| Construction | 2024–2027 |
| SPA | 1.25 mtpa, 20-year take-or-pay |
| Fixed payment | $2.00 + $0.50 × CPI factor per MMBtu |
| Commodity | 110% × Henry Hub |
| Gas supply | 101% × HH; 1 mtpa = 52 mm MMBtu |
| Transport | $0.20/MMBtu on **full capacity** |
| O&M | $50mm × CPI factor |
| Debt | 5.50% p.a., min DSCR 1.50x, amort from 2030 |

## 2. Answers

In [ ]:
from IPython.display import Markdown, display

display(Markdown(answers_markdown(result)))
for note in result.notes:
    print("-", note)

## 3. Cash-flow schedule

In [ ]:
df = pd.DataFrame({
    "year": result.years.astype(int),
    "cfads_mm": result.cfads / 1e6,
    "draw_mm": result.debt_draws / 1e6,
    "interest_mm": result.interest / 1e6,
    "amort_mm": result.amortization / 1e6,
    "debt_bal_mm": result.closing_balance / 1e6,
    "debt_service_mm": result.debt_service / 1e6,
    "dscr": result.dscr,
    "equity_cf_mm": result.equity_cashflow / 1e6,
})
display(df.round(2))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].bar(df["year"], df["cfads_mm"], color="#1f6f8b", label="CFADS")
axes[0].plot(df["year"], df["debt_service_mm"], color="#c44900", marker="o", label="Debt service")
axes[0].set_ylabel("$ mm")
axes[0].legend()
axes[0].set_title("CFADS vs debt service")

axes[1].plot(df["year"], df["debt_bal_mm"], color="#0b3c5d", marker="o")
axes[1].set_ylabel("$ mm")
axes[1].set_xlabel("Year")
axes[1].set_title("Debt balance")

plt.tight_layout()
plt.show()

## 4. What this notebook fixes

The original Excel template has several traps that break a naive Colab port:

1. **Production volume** — `Model!D10` pointed at the MMBtu conversion factor (`52e6`) instead of **SPA volume × conversion** (`1.25 × 52e6`).
2. **Fixed payment** — the memo is `$2.00 + $0.50 × CPI factor`, not `(2.00 + 0.50) × CPI factor`.
3. **Transport** — charged on **full 1.5 mtpa capacity**, not SPA volume.
4. **Sponsor IRR** — construction equity (`capex − debt draws`) must sit in the equity cash-flow series; the template’s “equity shortfall” row alone is not a full IRR bridge.
5. **Debt sizing** — sculpt amortization to 1.50x DSCR from 2030 and capitalize interest during construction so COD debt reflects IDC.

Re-run `run_model()` after changing assumptions in `src/lng_model.py`.